## Model Comparison

In [3]:
import os
import sys
sys.path.append('..')

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from src.hybrid_rag_pipeline import get_retriever
from src.context_builder import build_context
from src.prompts import prompt

In [4]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv('GROQ_API_KEY'))
models = client.models.list()
for m in models.data:
    print(m.id)

meta-llama/llama-prompt-guard-2-22m
groq/compound
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
llama-3.3-70b-versatile
meta-llama/llama-4-scout-17b-16e-instruct
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-v1-english
whisper-large-v3
llama-3.1-8b-instant
qwen/qwen3-32b
whisper-large-v3-turbo
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
groq/compound-mini
allam-2-7b


### New model: *qwen/qwen3-32b* vs Old model: *llama-3.1-8b-instant*

## Building Pipeline with both models

In [5]:
load_dotenv()

# load retriever
retriever = get_retriever(path='../')

# Model 1: current (Milestone 2)
llm_1 = ChatGroq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API_KEY"))

# Model 2: new
llm_2 = ChatGroq(model="qwen/qwen3-32b", api_key=os.getenv("GROQ_API_KEY"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index loaded from ../data/processed/faiss_index
Loading retriever...
retriever loaded from ../data/processed/bm25_index
Creating hybrid retriever...
Completed hybrid retriever!


In [6]:
def build_chain(llm):
    return (
        {"context": retriever | build_context, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

chain_1 = build_chain(llm_1)
chain_2 = build_chain(llm_2)

### Comparing 5 queries

In [7]:
# queries = [
#     "What Nintendo Switch games are good for young children?",
#     "Comfortable gaming headsets",
#     "What are some good racing games for Xbox?",
#     "Affordable wireless controller with good battery life",
#     "What gaming accessories would improve my setup under $50?"
# ]

In [8]:
query1 = "What Nintendo Switch games are good for young children?"

#original model llama
print(f"Query: {query1}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query1))

#new model qwen
print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query1))

Query: What Nintendo Switch games are good for young children?

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on the context, I don't have specific information about Nintendo Switch games for young children in the provided text. However, I do see a product related to the Nintendo Switch in the context: "nintendo mario kart 8  donkey kong video games for nintendo switch". 

While this product is not specifically mentioned as suitable for young children, I can make an educated guess based on the fact that Mario Kart series are often popular among children.

However, since there are no reviews specifically mentioning the suitability of this game for young children, I would like to recommend a different approach.

If you're looking for games suitable for young children, I would recommend checking the product descriptions and reviews of games on Amazon that are specifically designed for young children, such a

In [9]:
query2 = "Comfortable gaming headsets"

print(f"Query: {query2}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query2))

print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query2))

Query: Comfortable gaming headsets

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on the reviews and ratings provided, I would recommend the following comfortable gaming headsets:

1. 'Huhd 24Ghz Wireless Gaming Headset for Switch PC PS5 PS4 MacBook71 Surround Sound Headset for Computer with Detachable Noise Canceling Microphone' (Rating: 4.1/5) - This headset has been praised for its large size and comfort, making it a great option for users with bigger ears. Reviewer 'finally a large pair of headphones' mentioned that the headset fits great and is comfortable to wear.
2. 'Wired Gaming Headset with MicSades 709 35mm Stereo Over Ear Headphone for PC Laptop Notebook Tablet Smartphone by Afuntablue' (Rating: 3.3/5) - This headset has been praised for its comfort, with reviewers mentioning that it stays comfortable for long gaming sessions (up to 24 hours). Reviewer 'for a $20 gaming headset it was very com

In [10]:
query3 = "What are some good racing games for Xbox?"

print(f"Query: {query3}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query3))

print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query3))

Query: What are some good racing games for Xbox?

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on the context provided, I recommend the following racing games for Xbox:

1. **TrackMania Turbo** - This game has a rating of 4.4/5 and a price of $23.95. Reviewers have praised it as a "different type of racing" and a "must-try" for racing game enthusiasts. One reviewer mentioned it's a "great racing game" with over 200 head-spinning tracks.

2. **The Need for Speed** (Xbox One version is not available, but assuming you have a different console, this game is available) - While this game is not specifically mentioned as an Xbox title, it has a rating of 4.3/5 and is generally considered a classic in the racing game genre.

──────────────────────────────────────────────────
qwen/qwen3-32b
──────────────────────────────────────────────────
<think>
Okay, the user is asking for good racing games for Xbox. Let me 

In [11]:
query4 = "Affordable wireless controller with good battery life"

print(f"Query: {query4}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query4))

print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query4))

Query: Affordable wireless controller with good battery life

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on your query, I recommend the following affordable wireless controllers with good battery life:

1. **Feirsh Switch Controller Wireless Pro** - This controller has a rating of 4.4/5 and a price of $24.99. According to the reviews, it has a good battery life that lasts up to 7 days, and it's also comfortable to hold and easy to connect to the Nintendo Switch.

Review: "really good grip and bottomsbr very satisfied it is so good to play any game on the controller and the battery life will last 7 d"

2. **Etpark Switch Wireless Pro Controller** - Although its price is not specified, this controller has a rating of 3.9/5. According to one review, it has a good battery life and is easy to pair with the Nintendo Switch.

Review: "i love this controller  it is super easy to pair which is important becaus

In [12]:
query5 = "What gaming accessories would improve my setup under $50?"

print(f"Query: {query5}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query5))

print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query5))

Query: What gaming accessories would improve my setup under $50?

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on the provided context, I recommend the following gaming accessories that can improve your setup under $50:

1. **Eureka Ergonomic Call of Duty PC Computer Gaming Desk Setup Accessories Bundle**: This bundle is highly rated at 4.5/5 and priced at $29.99. It includes a headphone, controller stand, USB charger, unspillable drink cup holder, and power socket strip organizer. Reviewers praised its sturdy construction, ease of installation, and quality. Although the reviewer wished for a size-adjustable cup holder, it's a minor drawback.

2. **Enhance GXM3 Gaming Mouse**: While the price is not specified, I assume it's under $50 since it's not mentioned otherwise. The Enhance GXM3 Gaming Mouse has a rating of 3.6/5 and is suitable for competitive games like Dota 2, League of Legends, and World of W